> **Reproducing this notebook**
>
> 1. Run the pipeline to generate `walkforward_results_*.csv` and debug exports:
>    ```bash
>    python main.py
>    ```
> 2. Open this notebook (read-only on artifacts — no model re-training required).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from utils import (
    load_results_csv,
    load_equity_debug,
    plot_equity_vs_spikes,
    plot_selected_timeline,
)

# --- Figure style ---
mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.25
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False


# ---------------------------------------------------------------------------
# Notebook helper functions (defined once here)
# ---------------------------------------------------------------------------

def _ensure_datetime(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    df = df.copy()
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def slice_date_window(
    df: pd.DataFrame, start: str | None = None, end: str | None = None
) -> pd.DataFrame:
    """Slice a debug equity DataFrame by date strings, e.g. '2016-03-01'."""
    df = _ensure_datetime(df, "date")
    if start is not None:
        df = df[df["date"] >= pd.to_datetime(start)]
    if end is not None:
        df = df[df["date"] <= pd.to_datetime(end)]
    return df.copy()


def describe_spikes(df: pd.DataFrame) -> pd.DataFrame:
    """Summary of spike / near-spike rates and thresholds for a debug DataFrame."""
    df = _ensure_datetime(df, "date")
    n = len(df)
    spike_rate = float(np.mean(df["spike"])) if ("spike" in df.columns and n) else float("nan")
    near_rate = float(np.mean(df["near_spike"])) if ("near_spike" in df.columns and n) else float("nan")
    return pd.DataFrame([{
        "rows": n,
        "spike_rate": spike_rate,
        "near_spike_rate": near_rate,
        "vol_thr": float(df["vol_thr"].iloc[0]) if ("vol_thr" in df.columns and n) else float("nan"),
        "near_thr": float(df["near_thr"].iloc[0]) if ("near_thr" in df.columns and n) else float("nan"),
    }])


def preview_interesting_rows(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """Return rows where equity, selection, or spike flags change — more useful than head()."""
    df = _ensure_datetime(df, "date").copy()
    eq_dyn = df["equity_dynamic"].astype(float)
    eq_base = df["equity_baseline"].astype(float)
    changed = (
        df["selected_type"].ne(df["selected_type"].shift(1))
        | eq_dyn.ne(eq_dyn.shift(1))
        | eq_base.ne(eq_base.shift(1))
        | df["spike"].ne(df["spike"].shift(1))
        | df["near_spike"].ne(df["near_spike"].shift(1))
    )
    cols = ["date", "equity_baseline", "equity_dynamic", "selected_type",
            "signal_prev", "spike", "near_spike"]
    out = df.loc[changed, cols].copy()
    out["date"] = pd.to_datetime(out["date"]).dt.date
    return out.head(n)


def load_equity_debug_compat(pair: str, fold: int | None = None, fold_id: int | None = None) -> pd.DataFrame:
    """Accepts fold= or fold_id= to avoid call-site mistakes."""
    if fold is None and fold_id is None:
        raise ValueError("Pass fold=<int> or fold_id=<int>.")
    if fold is None:
        fold = fold_id
    return load_equity_debug(pair=pair, fold=int(fold))


---

# Part A — What / Why / How

---

# Regime-Aware Strategy Selection (Mixture-of-Experts Gating)
## Leakage-safe walk-forward evaluation + robustness mitigations

**One-liner:** A time-series ML system that routes between expert trading policies
(Trend Following vs Mean Reversion vs a rule-based baseline) using an XGBoost gating
model, evaluated with **leakage-safe** walk-forward backtesting and hardened against
two practical failure modes.

**What this notebook is:** an end-to-end engineering case study covering system design,
evaluation harness correctness, and failure-mode analysis.  
**What it is not:** a promise of profits.

---

## 1. Executive summary (quick skim)

**What this is:**  
A regime-aware **mixture-of-experts gating** system for a non-stationary time-series
decision problem. It routes between pre-defined expert strategy families
(Trend-Following, Mean-Reversion, and a rule-based router) using an **XGBoost classifier**,
evaluated with **leakage-safe walk-forward** backtesting.

**Why this matters (ML engineering angle):**
- In non-stationary settings, the "best" policy changes over time → the problem becomes
  **strategy selection** under changing conditions.
- The hard part is not training a model — it's building a **correct evaluation harness**
  (no leakage) and handling **failure modes** in online decision routing.

**Core approach:**
- **Rule-based regime labeling** (ADX + ATR%) for interpretable market phases.
- **Supervised gating model** (XGBoost) predicts which policy family is likely to
  perform best over a fixed horizon.
- **Online routing logic** adds practical controls:
  - confidence gating (τ_enter / τ_exit hysteresis)
  - minimum hold (reduce churn)
  - **volatility guard** (block MR during volatility spikes; threshold computed per fold
    on training data only)
  - **max-hold reset** (avoid lock-in; `max_hold_bars = 60`, applied only when flat)

**Primary evaluation (what to trust):**
- Expanding-window walk-forward evaluation across **14 FX pairs** (~20 years daily data).
- Fold-level comparisons against a strong baseline (**PhaseAware TF/MR**).

**Headline result snapshot (run46 default):**
- Dynamic selector vs PhaseAware baseline shows a **small but positive average uplift**
  in risk-adjusted performance.
- Improvements are not uniform — value comes from specific regimes; some folds still
  underperform (expected for gating systems).

**Artifacts produced:**
- Reproducible CSV outputs (per-fold, per-pair, summary tables)
- Debug equity / selection-timeline exports for failure-mode analysis

> This is a portfolio engineering project focused on demonstrating a robust ML +
> evaluation workflow on non-stationary time series — not a product pitch.


## 2. Problem framing: regime shifts → "which strategy should run next?"

Many real-world time series problems are **non-stationary**: the data-generating process
changes, and what works in one period can fail in another.

In trading terms:
- **Trend-following** tends to work better in persistent directional markets.
- **Mean-reversion** can work better in ranging / oscillatory markets.
- A single fixed strategy is often a poor fit across all regimes.

This project reframes the task as a **policy routing** problem:

> At each time step, given current context features, decide which "expert" policy family
> to execute next.

That is a classic **mixture-of-experts** pattern:
- **Experts** = hand-designed strategy families (TF, MR, and a rule-based PhaseAware router).
- **Gating model** = an XGBoost classifier that selects the expert family likely to perform
  best over a fixed horizon.
- **Evaluation** = walk-forward backtesting with strict leakage controls (because time-series
  leakage is easy to introduce accidentally).

The practical challenge is not only prediction accuracy — it's **robust decision-making**
under changing conditions:
- How do we avoid churn (over-switching)?
- How do we prevent catastrophic mistakes in rare conditions (e.g., volatility spikes)?
- How do we evaluate fairly without peeking into the future?

---

### 3. What is a "fold" in this notebook?

A **fold** is one out-of-sample evaluation step in an **expanding-window walk-forward** backtest.

For each fold:
1. **Train window:** fit preprocessing + train the gating model on all historical data up to time *t*.
2. **Test window:** run the trained selector on the next time slice (out-of-sample), recording
   strategy choices and backtest metrics.
3. **Step forward:** expand the training window and repeat.

Why fold-level analysis is the right unit:
- Each fold provides an independent mini out-of-sample experiment.
- It mimics periodic retraining in a live system.
- It reduces look-ahead bias — each fold's model and thresholds are **fit without any access
  to test-window data**.

In this notebook we report per-fold deltas (Dynamic − Baseline) for return / Sharpe / drawdown,
and use fold-level debug exports for failure-mode analysis.


---

# Part B — Evidence

---

## 1. Load artifacts

Three CSV files are read from `results/`, all produced by a single `python main.py` run.

| File | Unit | Contents |
|---|---|---|
| `walkforward_results_summary.csv` | run-level | Aggregate headline metrics across all folds and pairs |
| `walkforward_results_per_pair.csv` | pair-level | Mean metrics per instrument (averaged over folds) |
| `walkforward_results_per_fold.csv` | fold-level | Per-fold deltas — the primary analysis unit |

The fold-level file is the most important: it records how much Dynamic gained or lost versus
Baseline in each individual out-of-sample window, without any future-data contamination.


> **Leakage-safe evaluation note**
>
> All reported metrics come from walk-forward folds. For each fold, the gating model and
> any derived thresholds (e.g., the volatility-guard quantile) are fit on that fold's
> **training window only**, then evaluated on the subsequent **test window**.
>
> Fold-level analysis is the correct unit for non-stationary time series because it captures
> variation across time — a model that only wins in one long test period may be over-relying
> on a single regime. Fold distributions reveal whether gains are consistent or concentrated.


In [ ]:
summary = load_results_csv("walkforward_results_summary.csv")
per_pair = load_results_csv("walkforward_results_per_pair.csv")
per_fold = load_results_csv("walkforward_results_per_fold.csv")

summary


## 2. Results (global)

> **Metric conventions**
>
> - **Return Δ / Sharpe Δ / DD Δ** are computed as `Dynamic − Baseline` at the fold level.
> - **Max DD (%)** is stored as a **negative number** (e.g., −30 %).
> - Therefore **positive DD Δ = Dynamic had a less negative drawdown → better**.
> - Positive Sharpe Δ and Return Δ are straightforwardly better.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = per_fold["Sharpe Δ"].dropna().astype(float)

ax.hist(x, bins=40, alpha=0.85)
ax.axvline(0, color="black", linewidth=1)

ax.set_title("Figure 1 — Walk-forward distribution: Sharpe Δ (Dynamic − Baseline)")
ax.set_xlabel("Sharpe Δ")
ax.set_ylabel("Folds")

plt.tight_layout()
plt.show()


> **How to read Figure 1:** Each bar represents one walk-forward fold.
> A positive Sharpe Δ means the dynamic selector beat the baseline in that fold; negative
> means it underperformed.
>
> **Expected pattern:** Most folds cluster near 0 (many market windows look similar to
> the baseline). The goal is not to win every fold, but to gain edge in specific regimes
> without blowing up in others.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = per_fold["DD Δ"].dropna().astype(float)

ax.hist(x, bins=40, alpha=0.85)
ax.axvline(0, color="black", linewidth=1)

ax.set_title("Figure 2 — Walk-forward distribution: Max drawdown Δ (Dynamic − Baseline)")
ax.set_xlabel("DD Δ  (positive = Dynamic has lower drawdown / better)")
ax.set_ylabel("Folds")

plt.tight_layout()
plt.show()


> **How to read Figure 2:** With Max DD stored as a negative percentage,
> `DD Δ = Dynamic − Baseline > 0` means Dynamic was **less negative** → better.
>
> This plot is primarily a safety check: gains in Sharpe should not come from hidden
> catastrophic tail risk. A clean result shows the distribution skewed slightly positive
> or at least not materially negative.


In [ ]:
pp = per_pair.copy()
pp["Sharpe Δ"] = pp["Sharpe Δ"].astype(float)
pp_sorted = pp.sort_values("Sharpe Δ", ascending=False).reset_index(drop=True)

pairs = pp_sorted["Pair"].tolist()
vals  = pp_sorted["Sharpe Δ"].tolist()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pairs))

ax.bar(x, vals)
ax.axhline(0, color="black", linewidth=1)

ax.set_title("Figure 3 — Sharpe Δ by pair (Dynamic − Baseline, averaged over folds)")
ax.set_ylabel("Sharpe Δ")
ax.set_xticks(x)
ax.set_xticklabels(pairs, rotation=45, ha="right")

plt.tight_layout()
plt.show()


> **Per-pair results vary** — expected in non-stationary markets.
> This notebook uses a global selector configuration (run46: `max_hold_bars = 60`) that
> is a pragmatic compromise across all pairs, not tuned per instrument.
> The bar chart helps identify which pairs benefit most and which are neutral or mildly
> negative — useful for understanding where the gating logic adds value.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
df = per_fold.copy()
df["Sharpe Δ"] = df["Sharpe Δ"].astype(float)
df["DD Δ"]     = df["DD Δ"].astype(float)

ax.scatter(df["DD Δ"], df["Sharpe Δ"], s=12, alpha=0.35)
ax.axhline(0, color="black", linewidth=1)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Figure 3b — Fold-level trade-off: Sharpe Δ vs Max DD Δ")
ax.set_xlabel("DD Δ (Dynamic − Baseline)  (positive = less drawdown / better)")
ax.set_ylabel("Sharpe Δ (Dynamic − Baseline)")
plt.tight_layout()
plt.show()


---

# Part C — Failure Modes

---

## Case study: GBPJPY fold 8

This section uses the fold-level debug export (`results/equity_debug_GBPJPY_fold8.csv`) to
trace exactly why the selector behaves the way it does in one specific out-of-sample window.

### What happened

GBPJPY fold 8 covers a period that includes a sharp volatility spike (consistent with the
Brexit referendum in mid-2016). During and around this event, ATR% exceeded the fold's
training-derived spike threshold. Without a guard, the gating model could route to
Mean-Reversion precisely when adverse whipsaw moves are most likely — the worst possible
timing for a reversion strategy.

The volatility guard blocked MR selections during those bars, reducing the Dynamic selector's
exposure to the most dangerous conditions. The equity comparison in Figures 4 and 5 below
shows how the two paths diverged.

### Why it matters

This failure mode is subtle: the gating model's prediction is not wrong *per se*, but the
regime context has changed so abruptly that the historical training signal no longer holds.
A guard based on current volatility state provides a **model-free safety net** that complements
the learned gating probabilities.

Importantly, the spike threshold is computed from the **training slice only** (quantile of
ATR% on training bars). This makes the guard leakage-safe — the test-window volatility
structure never influences the threshold.

### What to look for in the figures below

- **Figure 4 (full fold):** overall shape of Dynamic vs Baseline equity; where do they diverge?
- **Figure 5 (zoomed window):** the specific period where spikes cluster; does Dynamic equity
  hold up better than Baseline? Does the drawdown panel show improvement?
- **Figure 6 (selection timeline):** are MR selections suppressed during red-shaded (spike)
  bars? How much switching is happening?
- **Table (after figures):** rows where selection, equity, or spike flags change — the
  granular trace of what the guard and selector were doing bar-by-bar.


> **What "spike" means in the plots**
>
> - `spike = True`: ATR% ≥ fold-specific threshold (computed on the training window only, quantile-based).
> - `near_spike = True`: ATR% is in an early-warning band (a multiple of the threshold).
>
> The threshold is **leakage-safe**: computed without using any test-window data.
> Red shading = spike region; orange shading = near-spike region.


In [ ]:
pair = "GBPJPY"
fold = 8

eq8 = load_equity_debug_compat(pair=pair, fold=fold)
eq8 = _ensure_datetime(eq8, "date")

# Zoom window of interest
zoom_start = "2016-03-01"
zoom_end   = "2016-08-31"
eq8_zoom   = slice_date_window(eq8, start=zoom_start, end=zoom_end)

# Spike / threshold summary for the zoom window
describe_spikes(eq8_zoom)


In [ ]:
# Figure 4 — Full fold: provides overall context before zooming in
plot_equity_vs_spikes(eq8, title=f"Figure 4 — Equity vs volatility spikes: {pair} fold={fold} (full fold)")
plt.tight_layout()
plt.show()


**Figure 4 — Full fold context.**  
Blue = Baseline equity; dark = Dynamic equity. Red shading = spike bars; orange = near-spike.
The lower panel shows drawdown (%) for both. Use this as orientation before the zoomed view.


In [ ]:
# --- Figure 5: zoom (narrow, actually different) ---
zoom_start = "2016-05-25"
zoom_end   = "2016-07-05"

eq8_zoom = slice_date_window(eq8, start=zoom_start, end=zoom_end)

eq8_zoom2 = eq8_zoom.copy()
eq8_zoom2["near_spike"] = False  # keep only spike shading for readability

plot_equity_vs_spikes(eq8_zoom2, f"Figure 5 — Equity vs spikes (spike-only shading): {pair} fold={fold}")
plt.tight_layout()
plt.show()


**Figure 5 — Zoomed equity comparison: what to observe**

**Baseline vs Dynamic paths:**  
In this window, the Baseline (PhaseAware rule-based router) is exposed to whatever
the regime model routes it to — including potential Mean-Reversion selections during
high-volatility bars. The Dynamic selector's volatility guard intervenes: when ATR%
exceeds the fold-specific spike threshold (red dashed line on the right axis), MR
selections are blocked.

**Volatility guard behavior:**  
The red-shaded bars mark spike events. During and around those bars, observe whether
the Dynamic equity path holds up better than Baseline (or at minimum avoids the
worst drawdown excursions visible in the lower panel).

**Leakage-safe thresholds:**  
The spike threshold is a training-window quantile of ATR%. It is computed **once per
fold, on training data only**, then applied to the test window. The test-window
volatility distribution never influences the threshold — this design choice makes
the guard evaluation honest.

**max_hold_bars context:**  
The current default (`max_hold_bars = 60`) means the selector will revert to PhaseAware
after 60 bars of continuous TF or MR selection — but only when the position is flat.
This prevents "lock-in" without interrupting active trades. This is a pragmatic
robustness knob, not an optimised global setting.


In [ ]:
# Figure 6 — Selected strategy timeline (same zoom window)
plot_selected_timeline(
    eq8_zoom,
    title=f"Figure 6 — Selected strategy timeline: {pair} fold={fold} ({zoom_start} to {zoom_end})",
)
plt.show()


**Figure 6 — Policy selection timeline.**  
The black step line shows which expert family was active each day.  
Blue dots mark switching events. Red shading = spike bars; orange = near-spike.

Key checks:
- Are MR selections (step = 1) suppressed during red-shaded periods? (volatility guard working)
- Is TF selected more often during spike regimes? (guard forcing a safer alternative)
- Does the selector thrash (many rapid switches)? (churn — mitigated by min-hold + hysteresis)
- Does it stay in one mode for a very long run? (lock-in — mitigated by `max_hold_bars = 60`)


In [ ]:
# Granular trace: rows where selection, equity, or spike flags change in the zoom window
preview_interesting_rows(eq8_zoom, n=30)


---

# Part D — Engineering Highlights + Next Steps

---

## Engineering highlights

**Pipeline discipline — leakage-safe preprocessing**  
Each walk-forward fold independently fits preprocessing transforms and derived thresholds
(volatility-guard quantile, scaling) on the training window. Nothing from the test window
bleeds back. This mirrors how a production system would be retrained periodically.

**Reproducibility artifacts**  
Every run produces versioned CSV outputs (per-fold, per-pair, summary). Run IDs (e.g.,
*run46*) allow comparing configurations systematically rather than relying on a single
end-to-end backtest.

**Debuggability**  
Fold-level equity debug exports (`equity_debug_{pair}_fold{fold}.csv`) record the
bar-by-bar equity for both Baseline and Dynamic alongside the spike mask, selection
state, and volatility feature. This makes it possible to answer "why did it do that?"
for any fold — essential for diagnosing production routing systems.

**Robustness mitigations grounded in failure analysis**  
- *Volatility guard:* identified that MR selection during extreme volatility produces
  concentrated losses → implemented a per-fold, training-only ATR% quantile threshold
  that blocks MR during spike regimes.
- *Max-hold reset:* identified that hysteresis + min-hold can cause prolonged lock-in to
  a non-default expert → implemented a time-based reset (`max_hold_bars = 60`) applied
  only when the position is flat, avoiding mid-trade interruptions.

**Note on `max_hold_bars`:**  
This parameter is a robustness knob. Different instruments and time windows have different
"sweet spots", and no single value is globally optimal. The value of 60 was chosen as a
pragmatic compromise that behaved well in aggregate walk-forward tests.

**Mixture-of-experts execution design**  
Hand-designed experts (TF, MR, PhaseAware) + ML gating + online routing logic
(confidence thresholds, hysteresis, min-hold, volatility guard, max-hold reset) form a
layered decision system where each layer addresses a distinct failure mode.

---

## Reasonable next steps

- **Switching costs / slippage:** model transaction costs explicitly so the selector
  can account for churn when making routing decisions.
- **Probability calibration:** calibrate gating model probabilities (e.g., Platt scaling,
  isotonic regression) and use them for soft position sizing or abstain logic.
- **Re-train frequency:** explore different walk-forward step sizes and model TTL policies
  to understand sensitivity to staleness.
- **Uncertainty / abstain:** add a "no-signal" / fallback option — when model confidence
  is low or regime is out-of-distribution, default to PhaseAware rather than forcing a choice.
- **Risk scaling:** use regime / volatility state to scale position size rather than only
  routing between binary expert selections.
